In [12]:
import json
import subprocess
import pandas as pd

def validate_jsonc_dataframe(
    df: pd.DataFrame,
    script_path: str = "/Users/shiyihe/Desktop/VIS_RESULT/validate_jsonc.js",
    participant_col: str = "participantId",
    format_col: str = "format",
    code_col: str = "code"
) -> pd.DataFrame:
    # 1. 准备输入
    records = df[[participant_col, format_col, code_col]].to_dict(orient="records")
    input_str = json.dumps(records)

    # 2. 调用脚本
    proc = subprocess.run(
        ["node", script_path],
        input=input_str.encode("utf-8"),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if proc.returncode != 0:
        raise RuntimeError(f"Node.js 校验脚本出错：{proc.stderr.decode()}")

    # 3. 解析输出并合并
    validated = json.loads(proc.stdout.decode("utf-8"))
    df_valid = pd.DataFrame(validated)

    # 保证列名一致
    df_valid.rename(columns={participant_col: participant_col}, inplace=True)

    # 4. 合并
    df_merged = df.merge(
        df_valid[[participant_col, "valid", "errorMessage"]],
        on=participant_col,
        how="left"
    )
    return df_merged


def summarize_jsonc_errors(
    df: pd.DataFrame,
    participant_col: str = "participantId",
    error_col: str = "errorMessage"
) -> pd.DataFrame:
    """
    对每个 participant 统计他们各自犯了哪些错误，以及每种错误出现了多少次。
    
    返回一个宽格式的 DataFrame，行索引是 participantId，
    列是不同的 errorType，值是出现次数。
    """
    # 1. 构造长格式列表
    records = []
    for _, row in df.iterrows():
        pid = row[participant_col]
        errs = row[error_col]
        if not errs or pd.isna(errs):
            continue
        # 拆分；再去掉“ at offset...” 的后缀
        for err in errs.split("; "):
            err_type = err.split(" at ")[0].strip()
            records.append({"participantId": pid, "errorType": err_type})
    
    if not records:
        return pd.DataFrame()  # 没记录就返回空表
    
    long_df = pd.DataFrame(records)
    
    # 2. 用 groupby + size 统计
    summary = (
        long_df
        .groupby([participant_col, "errorType"])
        .size()
        .reset_index(name="count")
    )
    
    # 3. 如果你想要宽格式（pivot table）：
    pivot = summary.pivot(
        index=participant_col,
        columns="errorType",
        values="count"
    ).fillna(0).astype(int)
    
    return pivot



In [ ]:
test_cases = [
    {
        "participantId": "test_case_1",
        "format": "jsonc",
        "code": '''{
  // standard key-value
  "name": "Shiyi",

  // number value with comment
  "age": 26,

  // nested object with trailing comma
  "details": {
    "city": "Salt Lake City",
    "student": true,
  },

  // trailing comma in array
  "hobbies": [
    "photography",
    "baking",
  ],
}''',
        "expected_valid": True
    },
    {
        "participantId": "test_case_2",
        "format": "jsonc",
        "code": '''{
  "valid": true // no trailing comma, still valid
}''',
        "expected_valid": True
    },
    {
        "participantId": "test_case_3",
        "format": "jsonc",
        "code": '''{
  "name": "Shiyi",
}''',
        "expected_valid": False
    },
    {
        "participantId": "test_case_4",
        "format": "jsonc",
        "code": '''{
  "nickname": 'neko'
}''',
        "expected_valid": False
    },
    {
        "participantId": "test_case_5",
        "format": "jsonc",
        "code": '''{
  "age": / this is a comment / 23
}''',
        "expected_valid": False
    },
    {
        "participantId": "test_case_6",
        "format": "jsonc",
        "code": '''{
  "a": 1
  "b": 2
}''',
        "expected_valid": False
    },
    {
        "participantId": "test_case_7",
        "format": "jsonc",
        "code": '''{
  "a": 1,''',
        "expected_valid": False
    },
    {
        "participantId": "test_case_8",
        "format": "jsonc",
        "code": '''{
  "x": 10,
  "y": 20,
}''',
        "expected_valid": True
    },
    {
        "participantId": "test_case_9",
        "format": "jsonc",
        "code": '''{
  "list": [
    1,
    2,
    3,
  ]
}''',
        "expected_valid": True
    },
    {
        "participantId": "test_case_10",
        "format": "jsonc",
        "code": '''{
  "a": 1,,
}''',
        "expected_valid": False
    }
]

# Create DataFrame
df_test_cases = pd.DataFrame(test_cases)

writing_config=pd.read_csv('/Users/shiyihe/Desktop/VIS_RESULT/not_use/writing_norm_config.csv')
writing_tabular=pd.read_csv('/Users/shiyihe/Desktop/VIS_RESULT/not_use/writing_norm_tabular.csv')
modifying_config=pd.read_csv('/Users/shiyihe/Desktop/VIS_RESULT/not_use/modifying_config.csv')
modifying_tabular=pd.read_csv('/Users/shiyihe/Desktop/VIS_RESULT/not_use/modifying_tabular.csv')
## writind jsonc
writing_config_jsonc=writing_config[['participantId','format','code']][writing_config['format']=='jsonc']
writing_tabular_jsonc=writing_tabular[['participantId','format','code']][writing_tabular['format']=='jsonc']
## modifying jsonc
modifying_config_jsonc=modifying_config[['participantId','task','format','code']][modifying_config['format']=='jsonc']
modifying_config_jsonc=modifying_config_jsonc.dropna(subset=['code'])
modifying_tabular_jsonc=modifying_tabular[['participantId','task','format','code']][modifying_tabular['format']=='jsonc']
modifying_tabular_jsonc=modifying_tabular_jsonc.dropna(subset=['code'])

In [ ]:
df_result=validate_jsonc_dataframe(df_test_cases)
df_result

,participantId,format,code,expected_valid,valid,errorMessage
0,test_case_1,jsonc,"{\n // standard key-value\n ""name"": ""Shiyi"",...",True,True,
1,test_case_2,jsonc,"{\n ""valid"": true // no trailing comma, still...",True,True,
2,test_case_3,jsonc,"{\n ""name"": ""Shiyi"",\n}",False,True,
3,test_case_4,jsonc,"{\n ""nickname"": 'neko'\n}",False,False,Invalid symbol at offset 16; Value expected at...
4,test_case_5,jsonc,"{\n ""age"": / this is a comment / 23\n}",False,False,Invalid symbol at offset 11; Invalid symbol at...
5,test_case_6,jsonc,"{\n ""a"": 1\n ""b"": 2\n}",False,False,Comma expected at offset 13
6,test_case_7,jsonc,"{\n ""a"": 1,",False,False,Property name expected at offset 11; Value exp...
7,test_case_8,jsonc,"{\n ""x"": 10,\n ""y"": 20,\n}",True,True,
8,test_case_9,jsonc,"{\n ""list"": [\n 1,\n 2,\n 3,\n ]\n}",True,True,
9,test_case_10,jsonc,"{\n ""a"": 1,,\n}",False,False,Property name expected at offset 11; Value exp...


In [17]:
df_result_mc = validate_jsonc_dataframe(modifying_config_jsonc)
df_result_mc[df_result_mc['valid']==True]

df_error_summary = summarize_jsonc_errors(df_result_mc)
df_error_summary

,participantId,task,format,code,valid,errorMessage
16,613a541cf948d295c1df8752,modifying-task-config-1,jsonc,"{\r\n ""name"": ""vega-lite"",\r\n ""author"": ""Do...",True,
17,613a541cf948d295c1df8752,modifying-task-config-1,jsonc,"{\r\n ""name"": ""vega-lite"",\r\n ""author"": ""Do...",True,
18,613a541cf948d295c1df8752,modifying-task-config-1,jsonc,"{\r\n ""name"": ""vega-lite"",\r\n ""author"": ""Do...",True,
20,613a541cf948d295c1df8752,modifying-task-config-2,jsonc,"{\r\n ""name"": ""vega-lite"",\r\n ""author"": ""Do...",True,
21,613a541cf948d295c1df8752,modifying-task-config-2,jsonc,"{\r\n ""name"": ""vega-lite"",\r\n ""author"": ""Do...",True,
...,...,...,...,...,...,...
155,67c6ed1e12728e4fe7a1edf6,modifying-task-config-4,jsonc,"{\r\n ""name"": ""vega-lite"",\r\n ""author"": ""Do...",True,
157,65298aca34ac384b783169d4,modifying-task-config-1,jsonc,"{\r\n ""name"": ""vega-lite"",\r\n ""author"": ""Do...",True,
161,65298aca34ac384b783169d4,modifying-task-config-2,jsonc,"{\r\n ""name"": ""vega-lite"",\r\n ""author"": ""Do...",True,
165,65298aca34ac384b783169d4,modifying-task-config-3,jsonc,"{\r\n ""name"": ""vega-lite"",\r\n ""author"": ""Do...",True,


In [25]:
# 检查具体的错误
pid = "57c357770e6a1f00015f6038"  # 要查询的 participantId

# 方法一：直接打印
# mask = df_result_wt["participantId"] == pid
# if mask.any():
#     print(df_result_wt.loc[mask, "errorMessage"].iloc[0])
# else:
#     print(f"No entry for participantId {pid}")

# 方法二：封装成函数
def print_error_message(df, pid):
    sub = df[df["participantId"] == pid]
    if sub.empty:
        print(f"No entry for participantId {pid}")
    else:
        print(f"Errors for {pid}:")
        print(sub["errorMessage"].iloc[0])

# 调用：
print_error_message(df_result_mc, pid)


Errors for 57c357770e6a1f00015f6038:
Comma expected at offset 207; Comma expected at offset 319


In [28]:
import demjson3
import pandas as pd

def parse_loose_jsonc(df, code_col="code"):
    """
    对 df[code_col] 中的字符串做超级宽松 JSONC/JSON5/HJSON 解析。
    返回：
      - parsed：解析结果或 None
      - parse_error：错误信息或 None
      - parsed_valid：True/False，表示是否成功解析
    """
    parsed, errors, valid = [], [], []
    for s in df[code_col]:
        try:
            parsed.append(demjson3.decode(s))
            errors.append(None)
            valid.append(True)
        except Exception as e:
            parsed.append(None)
            errors.append(str(e))
            valid.append(False)

    df2 = df.copy()
    df2["parsed"] = parsed
    df2["parse_error"] = errors
    df2["parsed_valid"] = valid
    return df2

# 使用示例
df_loose = parse_loose_jsonc(df_test_cases, code_col="code")
display(df_loose[["participantId","parsed_valid","parsed","parse_error"]])


,participantId,parsed_valid,parsed,parse_error
0,test_case_1,True,"{'name': 'Shiyi', 'age': 26, 'details': {'city...",None
1,test_case_2,True,{'valid': True},None
2,test_case_3,True,{'name': 'Shiyi'},None
3,test_case_4,True,{'nickname': 'neko'},None
4,test_case_5,False,None,Can not decode value starting with character '/'
5,test_case_6,False,None,Values must be separated by a comma
6,test_case_7,False,None,Object literal (dictionary) is not terminated
7,test_case_8,True,"{'x': 10, 'y': 20}",None
8,test_case_9,True,"{'list': [1, 2, 3]}",None
9,test_case_10,False,None,Can not omit elements of an object (dictionary)


In [29]:
import demjson3
import pandas as pd

def parse_loose_jsonc(df, code_col="code"):
    parsed, errors = [], []
    for s in df[code_col]:
        try:
            parsed.append(demjson3.decode(s))
            errors.append(None)
        except Exception as e:
            parsed.append(None)
            errors.append(str(e))

    df2 = df.copy()
    df2["parsed"] = parsed
    df2["parse_error"] = errors
    # 如果 parse_error 是 None 则解析成功
    df2["parsed_valid"] = df2["parse_error"].isna()
    return df2

# 使用示例同上
df_loose = parse_loose_jsonc(df_test_cases, code_col="code")
display(df_loose[["participantId","parsed_valid","parse_error"]])


,participantId,parsed_valid,parse_error
0,test_case_1,True,None
1,test_case_2,True,None
2,test_case_3,True,None
3,test_case_4,True,None
4,test_case_5,False,Can not decode value starting with character '/'
5,test_case_6,False,Values must be separated by a comma
6,test_case_7,False,Object literal (dictionary) is not terminated
7,test_case_8,True,None
8,test_case_9,True,None
9,test_case_10,False,Can not omit elements of an object (dictionary)


In [30]:
import pandas as pd

def summarize_loose_parse_errors(
    df: pd.DataFrame,
    participant_col: str = "participantId",
    error_col: str = "parse_error"
) -> pd.DataFrame:
    """
    对 df 中的 parse_error 列做宽松解析错误汇总。
    返回一个 pivot 表：行是 participantId，列是每种 errorType，值是次数。
    """
    records = []
    for _, row in df.iterrows():
        pid = row[participant_col]
        err = row[error_col]
        # 只统计真正出错的情况
        if pd.notna(err) and err:
            # demjson3.decode 出错时，err 就是一段消息，我们直接把它当作 errorType
            records.append({
                participant_col: pid,
                "errorType": err
            })
    if not records:
        return pd.DataFrame()  # 如果没有任何错误，返回空表

    long = pd.DataFrame(records)
    # 统计每个 participantId + errorType 的出现次数
    summary = (
        long
        .groupby([participant_col, "errorType"])
        .size()
        .reset_index(name="count")
    )
    # Pivot 成宽表
    pivot = summary.pivot(
        index=participant_col,
        columns="errorType",
        values="count"
    ).fillna(0).astype(int)
    return pivot

# 使用示例
df_loose_summary = summarize_loose_parse_errors(df_loose)
display(df_loose_summary)


errorType,Can not decode value starting with character '/',Can not omit elements of an object (dictionary),Object literal (dictionary) is not terminated,Values must be separated by a comma
participantId,,,,
test_case_10,0,1,0,0
test_case_5,1,0,0,0
test_case_6,0,0,0,1
test_case_7,0,0,1,0


In [35]:
import pandas as pd
import hjson
import demjson3
import re

def validate_hjson_strict(
    df: pd.DataFrame,
    code_col: str = "code",
    participant_col: str = "participantId"
) -> pd.DataFrame:
    """
    用 hjson 库对每条 code 做严格 HJSON 解析，
    返回新增两列：strict_valid (bool), strict_error (str or None)。
    """
    strict_valid = []
    strict_error = []
    for s in df[code_col]:
        try:
            hjson.loads(s)
            strict_valid.append(True)
            strict_error.append(None)
        except Exception as e:
            strict_valid.append(False)
            strict_error.append(str(e))
    df2 = df.copy()
    df2["strict_valid"] = strict_valid
    df2["strict_error"] = strict_error
    return df2

def summarize_hjson_errors(
    df: pd.DataFrame,
    participant_col: str = "participantId",
    error_col: str = "strict_error"
) -> pd.DataFrame:
    """
    将 strict_error 拆成 {participantId, errorType} 长表，
    pivot 成宽表：行 participantId，列 errorType，值为次数。
    """
    records = []
    for _, row in df.iterrows():
        pid = row[participant_col]
        err = row[error_col]
        if pd.notna(err) and err:
            records.append({participant_col: pid, "errorType": err})
    if not records:
        return pd.DataFrame()
    long = pd.DataFrame(records)
    summary = (
        long
        .groupby([participant_col, "errorType"])
        .size()
        .reset_index(name="count")
    )
    pivot = (
        summary
        .pivot(index=participant_col, columns="errorType", values="count")
        .fillna(0)
        .astype(int)
    )
    return pivot

def parse_loose_jsonc(
    df: pd.DataFrame,
    code_col: str = "code",
    participant_col: str = "participantId"
) -> pd.DataFrame:
    """
    用 demjson3 做超级宽松解析，新增 parsed_valid (bool), parse_error (str or None) 列。
    """
    parsed_valid = []
    parse_error = []
    for s in df[code_col]:
        try:
            demjson3.decode(s)
            parsed_valid.append(True)
            parse_error.append(None)
        except Exception as e:
            parsed_valid.append(False)
            parse_error.append(str(e))
    df2 = df.copy()
    df2["parsed_valid"] = parsed_valid
    df2["parse_error"] = parse_error
    return df2

def normalize_demjson_error(err: str) -> str:
    """
    将 demjson3 的错误信息归一化，只保留主要错误类别。
    """
    if pd.isna(err) or not err:
        return None

    # 常见几种模式
    if re.match(r"Unknown identifier\('.*?'\)", err):
        return "Unknown identifier"
    if "Missing value for object property" in err:
        return "Missing value for object property"
    if "Values must be separated by a comma" in err:
        return "Values must be separated by a comma"
    # 更多模式可以在这里添加……

    # 默认保留冒号前面部分
    return err.split(":")[0].strip()

def summarize_loose_errors_normalized(
    df: pd.DataFrame,
    participant_col: str = "participantId",
    error_col: str = "parse_error"
) -> pd.DataFrame:
    """
    对 demjson3 的 parse_error 先归一化，再 pivot 成宽表。
    """
    records = []
    for _, row in df.iterrows():
        pid = row[participant_col]
        err = row[error_col]
        e = normalize_demjson_error(err)
        if e:
            records.append({participant_col: pid, "errorType": e})
    if not records:
        return pd.DataFrame()
    long = pd.DataFrame(records)
    summary = long.groupby([participant_col, "errorType"]).size().reset_index(name="count")
    pivot = (
        summary
        .pivot(index=participant_col, columns="errorType", values="count")
        .fillna(0)
        .astype(int)
    )
    return pivot

# ——— 使用示例 ———
writing_tabular_hjson=writing_tabular[['participantId','format','code']][writing_tabular['format']=='hjson']

df_strict = validate_hjson_strict(writing_tabular_hjson)
strict_summary = summarize_hjson_errors(df_strict)

df_loose = parse_loose_jsonc(writing_tabular_hjson)
loose_summary = summarize_loose_errors_normalized(df_loose)

print("=== 严格 HJSON 校验错误汇总 ===")
display(strict_summary)

print("=== 宽松 DEMJSON3 解析错误汇总 ===")
display(loose_summary)


=== 严格 HJSON 校验错误汇总 ===


errorType,Extra data: line 12 column 1 - line 20 column 9 (char 183 - 315),Extra data: line 2 column 1 - line 17 column 20 (char 11 - 272),Extra data: line 2 column 5 - line 16 column 6 (char 17 - 261),Extra data: line 3 column 1 - line 13 column 9 (char 15 - 166),"Found ']' where a key name was expected (check your syntax or use quotes if the key name includes {}[],: or whitespace): line 25 column 12 (char 447)","Found '{' where a key name was expected (check your syntax or use quotes if the key name includes {}[],: or whitespace): line 2 column 1 (char 3)","Found '{' where a key name was expected (check your syntax or use quotes if the key name includes {}[],: or whitespace): line 6 column 25 (char 99)","Found '}' where a key name was expected (check your syntax or use quotes if the key name includes {}[],: or whitespace): line 1 column 2 (char 1)","Found '}' where a key name was expected (check your syntax or use quotes if the key name includes {}[],: or whitespace): line 2 column 1 (char 3)",Found whitespace in your key name (use quotes to include): line 2 column 4 (char 13)
participantId,,,,,,,,,,
56cb8858edf8da000b6df354,0,0,0,0,0,0,1,0,0,0
5c6414540821d30001046198,0,0,0,0,0,1,0,0,0,0
60a0b744298c64d46b63e893,0,0,0,0,0,0,0,1,0,0
61267b828ead584bcf092e35,0,0,0,0,0,0,0,0,1,0
6675c40cdc52b37294f0514e,0,0,0,0,1,0,0,0,0,0
667c18ace97ab6869cb1c50e,1,0,0,0,0,0,0,0,0,0
66cdfebcb908ea2717d06b6c,0,0,1,0,0,0,0,0,0,0
678f0bb28ec3307e4f1afb78,0,0,0,0,0,0,0,0,0,1
67adb27bd5f5776fcb16da62,0,0,0,1,0,0,0,0,0,0


=== 宽松 DEMJSON3 解析错误汇总 ===


errorType,"('Unknown identifier', 'John')","('Unknown identifier', 'Patients')","('Unknown identifier', 'john')","('Unknown identifier', 'patient')","('Unknown identifier', 'patients')",Missing value for object property,Values must be separated by a comma
participantId,,,,,,,
56cb8858edf8da000b6df354,1,0,0,0,0,0,0
5c6414540821d30001046198,0,0,0,0,0,1,0
60a0b744298c64d46b63e893,0,0,0,0,0,1,0
61267b828ead584bcf092e35,0,0,0,0,0,1,0
6577a27da9f5297d49f38e6f,0,0,0,0,0,0,1
65cf6d8a589a67afcab54e6f,0,0,1,0,0,0,0
6675c40cdc52b37294f0514e,1,0,0,0,0,0,0
667971a555af1f83d3f29ae7,1,0,0,0,0,0,0
667c18ace97ab6869cb1c50e,1,0,0,0,0,0,0
